In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    make_scorer,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

In [2]:
df = pd.read_csv("Cor_data.csv")

y = df["Cath"].map({
    "Cad": 1,
    "Normal": 0
})

X = df.drop(columns=["Cath"])

In [3]:
binary_cols = []

for col in X.columns:
    values = set(X[col].dropna().unique())

    if values.issubset({0, 1}):
        binary_cols.append(col)

In [4]:
numerical_cols = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

continuous_cols = [
    col for col in numerical_cols
    if col not in binary_cols
    and X[col].nunique() > 5
]

In [5]:
categorical_cols = X.select_dtypes(
    include=["object"]
).columns.tolist()

In [7]:
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

In [8]:
categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]
)

In [9]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, continuous_cols),
        ("cat", categorical_transformer, categorical_cols),
        ("bin", "passthrough", binary_cols)
    ]
)

In [10]:
log_reg = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    random_state=42
)

In [11]:
logistic_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", log_reg)
    ]
)

In [12]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [13]:
scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc"
}

In [14]:
results = cross_validate(
    logistic_pipeline,
    X,
    y,
    cv=cv,
    scoring=scoring,
    return_train_score=False
)

In [15]:
summary = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Sensitivity",
        "F1",
        "ROC-AUC"
    ],
    "Mean": [
        results["test_accuracy"].mean(),
        results["test_precision"].mean(),
        results["test_recall"].mean(),
        results["test_f1"].mean(),
        results["test_roc_auc"].mean()
    ],
    "Std": [
        results["test_accuracy"].std(),
        results["test_precision"].std(),
        results["test_recall"].std(),
        results["test_f1"].std(),
        results["test_roc_auc"].std()
    ]
})

summary

,Metric,Mean,Std
0,Accuracy,0.857760,0.053684
1,Precision,0.926510,0.029772
2,Sensitivity,0.869979,0.068618
3,F1,0.895947,0.040895
4,ROC-AUC,0.917890,0.038250


In [16]:
from sklearn.svm import SVC

svm = SVC(
    kernel="rbf",
    C=1.0,
    gamma="scale",
    probability=True,
    class_weight="balanced",
    random_state=42
)

svm_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", svm)
    ]
)

In [17]:
svm_results = cross_validate(
    svm_pipeline,
    X,
    y,
    cv=cv,
    scoring=scoring,
    return_train_score=False
)

In [18]:
svm_summary = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Sensitivity",
        "F1",
        "ROC-AUC"
    ],
    "Mean": [
        svm_results["test_accuracy"].mean(),
        svm_results["test_precision"].mean(),
        svm_results["test_recall"].mean(),
        svm_results["test_f1"].mean(),
        svm_results["test_roc_auc"].mean()
    ],
    "Std": [
        svm_results["test_accuracy"].std(),
        svm_results["test_precision"].std(),
        svm_results["test_recall"].std(),
        svm_results["test_f1"].std(),
        svm_results["test_roc_auc"].std()
    ]
})

svm_summary

,Metric,Mean,Std
0,Accuracy,0.861202,0.049922
1,Precision,0.926244,0.026335
2,Sensitivity,0.874736,0.060224
3,F1,0.898869,0.038720
4,ROC-AUC,0.907180,0.047627


In [19]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=300,
    class_weight="balanced",
    random_state=42
)

rf_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", rf)
    ]
)

In [20]:
rf_results = cross_validate(
    rf_pipeline,
    X,
    y,
    cv=cv,
    scoring=scoring,
    return_train_score=False
)

In [21]:
rf_summary = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Sensitivity",
        "F1",
        "ROC-AUC"
    ],
    "Mean": [
        rf_results["test_accuracy"].mean(),
        rf_results["test_precision"].mean(),
        rf_results["test_recall"].mean(),
        rf_results["test_f1"].mean(),
        rf_results["test_roc_auc"].mean()
    ],
    "Std": [
        rf_results["test_accuracy"].std(),
        rf_results["test_precision"].std(),
        rf_results["test_recall"].std(),
        rf_results["test_f1"].std(),
        rf_results["test_roc_auc"].std()
    ]
})

rf_summary

,Metric,Mean,Std
0,Accuracy,0.848087,0.043765
1,Precision,0.856045,0.040126
2,Sensitivity,0.948837,0.047433
3,F1,0.898891,0.029310
4,ROC-AUC,0.910801,0.044946


In [22]:
comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "SVM",
        "Random Forest"
    ],

    "Accuracy": [
        results["test_accuracy"].mean(),
        svm_results["test_accuracy"].mean(),
        rf_results["test_accuracy"].mean()
    ],

    "Precision": [
        results["test_precision"].mean(),
        svm_results["test_precision"].mean(),
        rf_results["test_precision"].mean()
    ],

    "Sensitivity": [
        results["test_recall"].mean(),
        svm_results["test_recall"].mean(),
        rf_results["test_recall"].mean()
    ],

    "F1": [
        results["test_f1"].mean(),
        svm_results["test_f1"].mean(),
        rf_results["test_f1"].mean()
    ],

    "ROC_AUC": [
        results["test_roc_auc"].mean(),
        svm_results["test_roc_auc"].mean(),
        rf_results["test_roc_auc"].mean()
    ]
})

comparison.sort_values(
    "ROC_AUC",
    ascending=False
)

,Model,Accuracy,Precision,Sensitivity,F1,ROC_AUC
0,Logistic Regression,0.857760,0.926510,0.869979,0.895947,0.917890
2,Random Forest,0.848087,0.856045,0.948837,0.898891,0.910801
1,SVM,0.861202,0.926244,0.874736,0.898869,0.907180


In [26]:
from sklearn.metrics import confusion_matrix

def specificity_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred
    ).ravel()

    return tn / (tn + fp)


def npv_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred
    ).ravel()

    return tn / (tn + fn)

In [27]:
scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "sensitivity": "recall",
    "specificity": make_scorer(specificity_score),
    "npv": make_scorer(npv_score),
    "f1": "f1",
    "roc_auc": "roc_auc",
    "pr_auc": "average_precision"
}

In [28]:
results = cross_validate(
    logistic_pipeline,
    X,
    y,
    cv=cv,
    scoring=scoring,
    return_train_score=False
)

In [29]:
summary = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Sensitivity",
        "Specificity",
        "NPV",
        "F1",
        "ROC-AUC",
        "PR-AUC"
    ],

    "Mean": [
        results["test_accuracy"].mean(),
        results["test_precision"].mean(),
        results["test_sensitivity"].mean(),
        results["test_specificity"].mean(),
        results["test_npv"].mean(),
        results["test_f1"].mean(),
        results["test_roc_auc"].mean(),
        results["test_pr_auc"].mean()
    ],

    "Std": [
        results["test_accuracy"].std(),
        results["test_precision"].std(),
        results["test_sensitivity"].std(),
        results["test_specificity"].std(),
        results["test_npv"].std(),
        results["test_f1"].std(),
        results["test_roc_auc"].std(),
        results["test_pr_auc"].std()
    ]
})

summary

,Metric,Mean,Std
0,Accuracy,0.857760,0.053684
1,Precision,0.926510,0.029772
2,Sensitivity,0.869979,0.068618
3,Specificity,0.827451,0.073552
4,NPV,0.735239,0.119220
5,F1,0.895947,0.040895
6,ROC-AUC,0.917890,0.038250
7,PR-AUC,0.964924,0.013865


In [30]:
svm_results = cross_validate(
    svm_pipeline,
    X,
    y,
    cv=cv,
    scoring=scoring,
    return_train_score=False
)

In [33]:
rf_results = cross_validate(
    rf_pipeline,
    X,
    y,
    cv=cv,
    scoring=scoring,
    return_train_score=False
)

In [35]:
comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "SVM",
        "Random Forest"
    ],

    "Accuracy": [
        results["test_accuracy"].mean(),
        svm_results["test_accuracy"].mean(),
        rf_results["test_accuracy"].mean()
    ],

    "Precision": [
        results["test_precision"].mean(),
        svm_results["test_precision"].mean(),
        rf_results["test_precision"].mean()
    ],

    "Sensitivity": [
        results["test_sensitivity"].mean(),
        svm_results["test_sensitivity"].mean(),
        rf_results["test_sensitivity"].mean()
    ],

    "Specificity": [
        results["test_specificity"].mean(),
        svm_results["test_specificity"].mean(),
        rf_results["test_specificity"].mean()
    ],

    "NPV": [
        results["test_npv"].mean(),
        svm_results["test_npv"].mean(),
        rf_results["test_npv"].mean()
    ],

    "F1": [
        results["test_f1"].mean(),
        svm_results["test_f1"].mean(),
        rf_results["test_f1"].mean()
    ],

    "ROC_AUC": [
        results["test_roc_auc"].mean(),
        svm_results["test_roc_auc"].mean(),
        rf_results["test_roc_auc"].mean()
    ],

    "PR_AUC": [
        results["test_pr_auc"].mean(),
        svm_results["test_pr_auc"].mean(),
        rf_results["test_pr_auc"].mean()
    ]
})

comparison.sort_values(
    "ROC_AUC",
    ascending=False
)

,Model,Accuracy,Precision,Sensitivity,Specificity,NPV,F1,ROC_AUC,PR_AUC
0,Logistic Regression,0.857760,0.926510,0.869979,0.827451,0.735239,0.895947,0.917890,0.964924
2,Random Forest,0.848087,0.856045,0.948837,0.600654,0.846275,0.898891,0.910801,0.954370
1,SVM,0.861202,0.926244,0.874736,0.828105,0.738421,0.898869,0.907180,0.954317


In [36]:
from sklearn.model_selection import cross_val_predict

logistic_probs = cross_val_predict(
    logistic_pipeline,
    X,
    y,
    cv=cv,
    method="predict_proba"
)[:, 1]

In [37]:
svm_probs = cross_val_predict(
    svm_pipeline,
    X,
    y,
    cv=cv,
    method="predict_proba"
)[:, 1]

In [38]:
rf_probs = cross_val_predict(
    rf_pipeline,
    X,
    y,
    cv=cv,
    method="predict_proba"
)[:, 1]